In [ ]:
# Test ConsensusLeidenClustering
import igraph as ig
import numpy as np
import pandas as pd
import scipy.sparse as sp
from skclust.graph import ConsensusLeidenClustering, cluster_membership_cooccurrence

# ============================================================================
# Setup: Create test graph
# ============================================================================
graph = ig.Graph.Famous('Zachary')
graph.vs['name'] = [f'node_{i}' for i in range(graph.vcount())]
np.random.seed(42)
graph.es['weight'] = np.random.uniform(0.1, 1.0, graph.ecount())

print(f"Test graph: {graph.vcount()} nodes, {graph.ecount()} edges")

print("\n" + "=" * 80)
print("TEST 1: Basic functionality with verbose output")
print("=" * 80)

leiden = ConsensusLeidenClustering(n_iter=10, n_jobs=1, random_state=42, verbose=2)
leiden.fit(graph)

print(f"\nPartitions shape: {leiden.partitions_.shape}")
print(f"Membership matrix shape: {leiden.membership_matrix_.shape}")
print(f"Consensus edges: {len(leiden.consensus_edges_)} / {graph.ecount()} edges")
print(f"Consensus ratio: mean={leiden.consensus_ratio_.mean():.3f}, median={leiden.consensus_ratio_.median():.3f}")

labels = leiden.transform(graph)
print(f"\nCluster labels:\n{labels.value_counts()}")
print(f"Number of clusters: {leiden.n_clusters_}")
print(f"Discarded nodes: {len(leiden.discarded_nodes_)}")
print(f"Unstable nodes: {len(leiden.unstable_nodes_)}")
print(f"Consensus graph: {leiden.consensus_graph_.vcount()} nodes, {leiden.consensus_graph_.ecount()} edges")
print(f"Discarded consensus graph: {leiden.consensus_graph_discarded_.vcount()} nodes, {leiden.consensus_graph_discarded_.ecount()} edges")
print(f"Filtered graph: {leiden.filtered_graph_.vcount()} nodes, {leiden.filtered_graph_.ecount()} edges")
print(f"\nModularity:\n{leiden.modularity_}")
print(f"\nStability report:\n{leiden.summary_}")

print("\n" + "=" * 80)
print("TEST 2: Resolution parameter sweep")
print("=" * 80)

for res in [0.5, 1.0, 1.5, 2.0]:
    leiden_res = ConsensusLeidenClustering(
        n_iter=10, resolution_parameter=res, n_jobs=-1, random_state=42, verbose=0
    )
    leiden_res.fit(graph)
    n_clusters_per_iter = leiden_res.partitions_.nunique(axis=0)
    print(f"Resolution {res}: "
          f"consensus edges = {len(leiden_res.consensus_edges_):3d}, "
          f"clusters = {leiden_res.n_clusters_:2d}, "
          f"avg leiden clusters = {n_clusters_per_iter.mean():.1f}, "
          f"modularity(filtered) = {leiden_res.modularity_['filtered']:.3f}")

print("\n" + "=" * 80)
print("TEST 3: Weighted graph")
print("=" * 80)

leiden_weighted = ConsensusLeidenClustering(
    n_iter=10, weight='weight', n_jobs=-1, random_state=42, verbose=0
)
labels_weighted = leiden_weighted.fit_transform(graph)

print(f"Weighted consensus edges: {len(leiden_weighted.consensus_edges_)} / {graph.ecount()}")
print(f"Weighted mean consensus: {leiden_weighted.consensus_ratio_.mean():.3f}")
print(f"Weighted clusters: {leiden_weighted.n_clusters_}")
print(f"Weighted modularity: {leiden_weighted.modularity_['filtered']:.3f}")

print("\n" + "=" * 80)
print("TEST 4: Different partition types")
print("=" * 80)

from leidenalg import ModularityVertexPartition, CPMVertexPartition

leiden_mod = ConsensusLeidenClustering(
    n_iter=10, partition_type=ModularityVertexPartition, n_jobs=-1, random_state=42, verbose=0
)
labels_mod = leiden_mod.fit_transform(graph)
print(f"ModularityVertexPartition: {len(leiden_mod.consensus_edges_)} consensus edges, {leiden_mod.n_clusters_} clusters")

leiden_cpm = ConsensusLeidenClustering(
    n_iter=10, partition_type=CPMVertexPartition, leiden_kws={'resolution_parameter': 0.1},
    n_jobs=-1, random_state=42, verbose=0
)
labels_cpm = leiden_cpm.fit_transform(graph)
print(f"CPMVertexPartition: {len(leiden_cpm.consensus_edges_)} consensus edges, {leiden_cpm.n_clusters_} clusters")

print("\n" + "=" * 80)
print("TEST 5: cluster_membership_cooccurrence and sparse membership matrix")
print("=" * 80)

df_partitions = leiden.partitions_
print(f"Input partitions shape: {df_partitions.shape}")

# All-pairs path
cooccur_all = cluster_membership_cooccurrence(df_partitions)
n_expected_pairs = (graph.vcount() * (graph.vcount() - 1)) // 2
assert cooccur_all.shape[0] == n_expected_pairs, f"Expected {n_expected_pairs} pairs, got {cooccur_all.shape[0]}"
print(f"✓ All-pairs: {cooccur_all.shape[0]} pairs (expected {n_expected_pairs})")

# Edge-list path
edge_list = [frozenset([graph.vs[e.source]['name'], graph.vs[e.target]['name']]) for e in graph.es]
cooccur_edges = cluster_membership_cooccurrence(df_partitions, edge_list=edge_list)
assert cooccur_edges.shape[0] == graph.ecount(), f"Expected {graph.ecount()} edges, got {cooccur_edges.shape[0]}"
print(f"✓ Edge-list path: {cooccur_edges.shape[0]} edges")

# Verify membership_matrix_ is sparse
assert isinstance(leiden.membership_matrix_, sp.csr_matrix), "membership_matrix_ must be scipy.sparse.csr_matrix"
print(f"✓ membership_matrix_ is csr_matrix: shape={leiden.membership_matrix_.shape}")

# Verify get_membership_matrix returns sparse DataFrame matching original computation
df_membership = leiden.get_membership_matrix()
assert isinstance(df_membership, pd.DataFrame), "get_membership_matrix must return pd.DataFrame"
assert all(isinstance(dt, pd.SparseDtype) for dt in df_membership.dtypes), "get_membership_matrix must use SparseDtype"
assert np.array_equal(cooccur_edges.values, df_membership.values), "get_membership_matrix does not match standalone computation"
print(f"✓ get_membership_matrix: sparse DataFrame matches standalone computation")

print("\n" + "=" * 80)
print("TEST 6: Parallel vs Sequential consistency")
print("=" * 80)

leiden_seq = ConsensusLeidenClustering(n_iter=10, n_jobs=1, random_state=999, verbose=0)
labels_seq = leiden_seq.fit_transform(graph)

leiden_par = ConsensusLeidenClustering(n_iter=10, n_jobs=-1, random_state=999, verbose=0)
labels_par = leiden_par.fit_transform(graph)

assert leiden_seq.partitions_.equals(leiden_par.partitions_), "Sequential != Parallel partitions!"
assert set(leiden_seq.consensus_edges_) == set(leiden_par.consensus_edges_), "Sequential != Parallel consensus!"
assert labels_seq.equals(labels_par), "Sequential != Parallel labels!"
print("✓ Sequential and parallel execution produce identical results")

print("\n" + "=" * 80)
print("TEST 7: fit_transform vs fit_predict")
print("=" * 80)

leiden_ft = ConsensusLeidenClustering(n_iter=10, n_jobs=-1, random_state=42, verbose=0)
labels_ft = leiden_ft.fit_transform(graph)

leiden_fp = ConsensusLeidenClustering(n_iter=10, n_jobs=-1, random_state=42, verbose=0)
labels_fp = leiden_fp.fit_predict(graph)

assert labels_ft.equals(labels_fp), "fit_transform != fit_predict!"
print("✓ fit_transform and fit_predict produce identical results")

print("\n" + "=" * 80)
print("TEST 8: minimum_cluster_size filtering and node categories")
print("=" * 80)

leiden_min1 = ConsensusLeidenClustering(
    n_iter=10, minimum_cluster_size=1, n_jobs=-1, random_state=42, verbose=0
)
leiden_min1.fit(graph)

leiden_min7 = ConsensusLeidenClustering(
    n_iter=10, minimum_cluster_size=7, n_jobs=-1, random_state=42, verbose=0
)
leiden_min7.fit(graph)

print(f"min_size=1: {leiden_min1.n_clusters_} clusters, {len(leiden_min1.labels_)} nodes, {len(leiden_min1.discarded_nodes_)} discarded")
print(f"  unstable={len(leiden_min1.unstable_nodes_)}")
print(f"min_size=7: {leiden_min7.n_clusters_} clusters, {len(leiden_min7.labels_)} nodes, {len(leiden_min7.discarded_nodes_)} discarded")
print(f"  unstable={len(leiden_min7.unstable_nodes_)}")

# More nodes should be discarded with higher minimum
assert len(leiden_min7.discarded_nodes_) >= len(leiden_min1.discarded_nodes_), "Higher min_size should discard more nodes"
assert leiden_min7.n_clusters_ <= leiden_min1.n_clusters_, "Higher min_size should have fewer clusters"

# Verify labels + discarded + unstable = total
assert len(leiden_min7.labels_) + len(leiden_min7.discarded_nodes_) + len(leiden_min7.unstable_nodes_) == graph.vcount(), "labels + discarded + unstable must equal total nodes"
print("✓ labels + discarded + unstable = total nodes")

# Verify unstable and discarded don't overlap
assert len(set(leiden_min7.unstable_nodes_) & set(leiden_min7.discarded_nodes_)) == 0, "unstable and discarded must not overlap"
print("✓ unstable and discarded nodes do not overlap")

# Verify filtered_graph_ has correct node count
assert leiden_min7.filtered_graph_.vcount() == len(leiden_min7.labels_), "filtered_graph_ node count must match labels"
print("✓ filtered_graph_ node count matches labels")

# Filtered modularity should differ from initial when clusters are actually dropped
assert leiden_min7.modularity_['initial'] != leiden_min7.modularity_['filtered'], "Modularity should differ when clusters are dropped"
print("✓ initial and filtered modularity differ when clusters are dropped")

# Verify modularity series order and keys
assert list(leiden_min7.modularity_.index) == ['initial', 'filtered', 'consensus'], "modularity_ order must be [initial, filtered, consensus]"
print(f"✓ Modularity order correct: {list(leiden_min7.modularity_.index)}")
print(f"  initial={leiden_min7.modularity_['initial']:.3f}, filtered={leiden_min7.modularity_['filtered']:.3f}, consensus={leiden_min7.modularity_['consensus']:.3f}")

print("\n" + "=" * 80)
print("TEST 9: consensus_threshold")
print("=" * 80)

leiden_strict = ConsensusLeidenClustering(
    n_iter=50, consensus_threshold=1.0, n_jobs=-1, random_state=42, verbose=0
)
leiden_strict.fit(graph)

leiden_relaxed = ConsensusLeidenClustering(
    n_iter=50, consensus_threshold=0.8, n_jobs=-1, random_state=42, verbose=0
)
leiden_relaxed.fit(graph)

print(f"threshold=1.0: {len(leiden_strict.consensus_edges_)} consensus edges, {leiden_strict.n_clusters_} clusters")
print(f"threshold=0.8: {len(leiden_relaxed.consensus_edges_)} consensus edges, {leiden_relaxed.n_clusters_} clusters")

assert len(leiden_relaxed.consensus_edges_) >= len(leiden_strict.consensus_edges_), "Relaxed threshold should have more edges"
print("✓ Lower threshold retains more edges")

print("\n" + "=" * 80)
print("TEST 10: get_feature_names_out and consensus_edges_")
print("=" * 80)

# consensus_edges_ should be pd.Index
assert isinstance(leiden.consensus_edges_, pd.Index), "consensus_edges_ must be pd.Index"
assert leiden.consensus_edges_.name == "Edge", "consensus_edges_ name must be 'Edge'"
assert all(isinstance(e, frozenset) for e in leiden.consensus_edges_), "All entries must be frozenset"
print(f"✓ consensus_edges_: pd.Index with {len(leiden.consensus_edges_)} frozenset edges")

# get_feature_names_out returns the same object
edges_idx = leiden.get_feature_names_out()
assert edges_idx is leiden.consensus_edges_, "get_feature_names_out must return consensus_edges_ directly"
print("✓ get_feature_names_out returns consensus_edges_ directly")

print("\n" + "=" * 80)
print("TEST 11: Error handling")
print("=" * 80)

leiden_error = ConsensusLeidenClustering()

try:
    leiden_error.transform(graph)
    print("✗ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"✓ transform before fit: {e}")

try:
    leiden_error.get_feature_names_out()
    print("✗ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"✓ get_feature_names_out before fit: {e}")

try:
    leiden_error.get_membership_matrix()
    print("✗ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"✓ get_membership_matrix before fit: {e}")

graph_no_name = ig.Graph.Famous('Zachary')
try:
    leiden_error.fit(graph_no_name)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Missing 'name' attribute: {e}")

leiden_bad_weight = ConsensusLeidenClustering(weight='nonexistent')
try:
    leiden_bad_weight.fit(graph)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Invalid weight attribute: {e}")

try:
    ConsensusLeidenClustering(consensus_threshold=1.5).fit(graph)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Invalid consensus_threshold: {e}")

try:
    ConsensusLeidenClustering(minimum_cluster_size=0).fit(graph)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Invalid minimum_cluster_size: {e}")

print("\n" + "=" * 80)
print("TEST 12: Custom cluster prefix, types, and labels structure")
print("=" * 80)

leiden_custom = ConsensusLeidenClustering(
    n_iter=10, cluster_prefix="community_", n_jobs=-1, random_state=42, verbose=0
)
labels_custom = leiden_custom.fit_transform(graph)

assert all(l.startswith("community_") for l in labels_custom.values), "All labels must use custom prefix"
assert labels_custom.isna().sum() == 0, "labels_ should not contain NaN"
assert labels_custom.index.name == "Node", "Index name must be 'Node'"
assert labels_custom.name == "Cluster", "Series name must be 'Cluster'"
print(f"✓ Custom prefix, no NaN, correct names:\n{labels_custom.value_counts().head()}")

# Verify discarded_nodes_ type
assert isinstance(leiden_custom.discarded_nodes_, pd.Index), "discarded_nodes_ must be pd.Index"
assert leiden_custom.discarded_nodes_.name == "Node", "discarded_nodes_ name must be 'Node'"
print(f"✓ discarded_nodes_: pd.Index with {len(leiden_custom.discarded_nodes_)} entries")

# Verify unstable_nodes_ type
assert isinstance(leiden_custom.unstable_nodes_, pd.Index), "unstable_nodes_ must be pd.Index"
assert leiden_custom.unstable_nodes_.name == "Node", "unstable_nodes_ name must be 'Node'"
print(f"✓ unstable_nodes_: pd.Index with {len(leiden_custom.unstable_nodes_)} entries")

# Verify summary_ type and fields
assert isinstance(leiden_custom.summary_, pd.Series), "summary_ must be pd.Series"
assert leiden_custom.summary_.name == "Summary", "summary_ name must be 'Summary'"
expected_fields = {
    # Graph sizes
    'n_nodes_initial', 'n_edges_initial',
    'n_nodes_filtered', 'n_edges_filtered',
    'n_edges_consensus',
    # Clustering
    'n_clusters', 'n_unstable', 'n_discarded',
    # Consensus
    'n_consensus_edges', 'consensus_ratio', 'consensus_std',
    'pct_100', 'pct_90plus', 'pct_80plus', 'pct_70plus', 'pct_60plus', 'pct_below_50',
    # Modularity
    'modularity_initial', 'modularity_filtered', 'modularity_consensus',
}
assert set(leiden_custom.summary_.index) == expected_fields, f"summary_ fields mismatch: {set(leiden_custom.summary_.index)}"
print(f"✓ summary_: pd.Series with correct fields")

# Verify consensus_ratio_ name
assert leiden_custom.consensus_ratio_.name == "ConsensusRatio", "consensus_ratio_ name must be 'ConsensusRatio'"
print(f"✓ consensus_ratio_: name='ConsensusRatio'")

# Verify modularity_ type
assert isinstance(leiden_custom.modularity_, pd.Series), "modularity_ must be pd.Series"
assert leiden_custom.modularity_.name == "Modularity", "modularity_ name must be 'Modularity'"
assert list(leiden_custom.modularity_.index) == ['initial', 'filtered', 'consensus'], "modularity_ order wrong"
print(f"✓ modularity_: pd.Series with correct order")

print("\n" + "=" * 80)
print("ALL TESTS PASSED ✓")
print("=" * 80)

2026-02-25 13:41:20.307 | INFO     | skclust.graph:_log:378 - Validating input graph
2026-02-25 13:41:20.307 | INFO     | skclust.graph:_log:378 - Graph: 34 nodes, 78 edges
2026-02-25 13:41:20.314 | INFO     | skclust.graph:_log:378 - Using 1 parallel jobs
2026-02-25 13:41:20.314 | INFO     | skclust.graph:_log:378 - Running 10 Leiden iterations


Test graph: 34 nodes, 78 edges

TEST 1: Basic functionality with verbose output


Leiden clustering:   0%|          | 0/10 [00:00<?, ?it/s]

2026-02-25 13:41:20.326 | INFO     | skclust.graph:_log:378 - Leiden iterations completed in 0.01s
2026-02-25 13:41:20.328 | INFO     | skclust.graph:_log:378 - Computing cluster membership co-occurrence matrix
2026-02-25 13:41:20.329 | INFO     | skclust.graph:_log:378 - Co-occurrence matrix: (78, 10), computed in 0.00s
2026-02-25 13:41:20.330 | INFO     | skclust.graph:_log:378 - Found 57 consensus edges (>= 1.0 threshold)
2026-02-25 13:41:20.331 | INFO     | skclust.graph:_log:378 - Building consensus graph


Filtering consensus edges:   0%|          | 0/78 [00:00<?, ?it/s]

2026-02-25 13:41:20.333 | INFO     | skclust.graph:_log:378 - Consensus graph (pre-filter): 34 nodes, 57 edges (0.00s)
2026-02-25 13:41:20.334 | INFO     | skclust.graph:_log:378 - Computing connected components for cluster labels
2026-02-25 13:41:20.335 | INFO     | skclust.graph:_log:378 - Found 4 clusters in 0.00s
2026-02-25 13:41:20.335 | INFO     | skclust.graph:_log:378 - Building final graphs and computing modularity
2026-02-25 13:41:20.336 | INFO     | skclust.graph:_log:378 - Consensus graph: 34 nodes, 57 edges
2026-02-25 13:41:20.336 | INFO     | skclust.graph:_log:378 - Filtered graph: 34 nodes, 78 edges
2026-02-25 13:41:20.336 | INFO     | skclust.graph:_log:378 - Modularity (initial=0.4198, consensus=0.6753, filtered=0.4198)
2026-02-25 13:41:20.337 | INFO     | skclust.graph:_log:378 - Total fit time: 0.03s
2026-02-25 13:41:20.337 | INFO     | skclust.graph:fit:709 - ============================================================
2026-02-25 13:41:20.337 | INFO     | skclust.g


Partitions shape: (34, 10)
Membership matrix shape: (78, 10)
Consensus edges: 57 / 78 edges
Consensus ratio: mean=0.731, median=1.000

Cluster labels:
Cluster
leiden_1    12
leiden_2    11
leiden_3     6
leiden_4     5
Name: count, dtype: int64
Number of clusters: 4


AttributeError: 'ConsensusLeidenClustering' object has no attribute 'excluded_nodes_'

In [ ]:
leiden_min7.modularity_
# initial      0.280819
# filtered     0.361976
# consensus    0.468298
# Name: Modularity, dtype: float64

In [ ]:
leiden_min7.summary_

In [ ]:
leiden_min7.consensus_graph_
